In [9]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_log_error, r2_score

print('Setup complete')

Setup complete


In [10]:
CONTINUOUS_FEATURES= ['LotArea', 'YearBuilt', 'OverallQual', 'OverallCond', 'GrLivArea', 'TotalBsmtSF', 'GarageArea', '1stFlrSF']
CATEGORICAL_FEATURES = ['MSZoning', 'Neighborhood', 'BldgType', 'HouseStyle', 'SaleCondition', 'SaleType']
FEATURE_COLUMNS = CONTINUOUS_FEATURES + CATEGORICAL_FEATURES
LABEL_COLUMN = 'SalePrice'
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(exist_ok=True)

print('Constants defined')


Constants defined


In [3]:
import os
print(os.getcwd())

C:\Users\Rama\Desktop\dsp-bhanu-kuncham\notebooks


In [4]:
raw_df = pd.read_csv('../data/train.csv').dropna(subset=[LABEL_COLUMN])
X = raw_df[FEATURE_COLUMNS]
y = raw_df[LABEL_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows')

Train: 1168 rows | Test: 292 rows


In [8]:
X_train = X_train.copy()

# fill missing - continuous
for col in CONTINUOUS_FEATURES:
    X_train[col] = X_train[col].fillna(X_train[col].mean())

# fill missing - categorical  
for col in CATEGORICAL_FEATURES:
    X_train[col] = X_train[col].fillna(X_train[col].mode()[0])

# scaler - fit then save then transform
scaler = StandardScaler()
scaler.fit(X_train[CONTINUOUS_FEATURES])
joblib.dump(scaler, MODEL_DIR / 'scaler.joblib')
scaled_train = pd.DataFrame(
    scaler.transform(X_train[CONTINUOUS_FEATURES]),
    columns=CONTINUOUS_FEATURES,
    index=X_train.index
)

# encoder - fit then save then transform
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder.fit(X_train[CATEGORICAL_FEATURES])
joblib.dump(encoder, MODEL_DIR / 'encoder.joblib')
encoded_cols = encoder.get_feature_names_out(CATEGORICAL_FEATURES)
encoded_train = pd.DataFrame(
    encoder.transform(X_train[CATEGORICAL_FEATURES]),
    columns=encoded_cols,
    index=X_train.index
)
X_train_processed = pd.concat([scaled_train, encoded_train], axis=1)
print('Train preprocessing done')

Train preprocessing done


In [13]:
# create the model
model = LinearRegression()

# train it on processed train data
model.fit(X_train_processed, y_train)

# save it to disk
joblib.dump(model, MODEL_DIR / 'model.joblib')

print(f'Model trained on {X_train.shape[0]} samples and saved')

Model trained on 1168 samples and saved


In [11]:
# load saved transformers from disk
scaler = joblib.load(MODEL_DIR / 'scaler.joblib')
encoder = joblib.load(MODEL_DIR / 'encoder.joblib')

# fill missing values on X_test
X_test = X_test.copy()
for col in CONTINUOUS_FEATURES:
    X_test[col] = X_test[col].fillna(X_test[col].mean())
for col in CATEGORICAL_FEATURES:
    X_test[col] = X_test[col].fillna(X_test[col].mode()[0])

# transform only - no fit!
scaled_test = pd.DataFrame(
    scaler.transform(X_test[CONTINUOUS_FEATURES]),
    columns=CONTINUOUS_FEATURES,
    index=X_test.index
)
encoded_test = pd.DataFrame(
    encoder.transform(X_test[CATEGORICAL_FEATURES]),
    columns=encoder.get_feature_names_out(CATEGORICAL_FEATURES),
    index=X_test.index
)

X_test_processed = pd.concat([scaled_test,encoded_test], axis=1)
print('Test preprocessing done')

Test preprocessing done


In [25]:
# load the saved model
model = joblib.load(MODEL_DIR / 'model.joblib')

# make predictions on test set
y_pred = model.predict(X_test_processed)

# clip negative predictions to 1 (prices can't be negative)
y_pred = np.maximum(y_pred, 1)

# calculate metrics
rmsle = round(float(np.sqrt(mean_squared_log_error(y_test, y_pred))), 4)
r2 = round(float(r2_score(y_test, y_pred)), 4)

print(f'RMSLE: {rmsle} | R2: {r2}')

RMSLE: 0.1745 | R2: 0.84


In [7]:
# load inference data
inference_df = pd.read_csv('../data/test.csv')[FEATURE_COLUMNS].copy()

# load saved transformers and model from disk
scaler = joblib.load(MODEL_DIR / 'scaler.joblib')
encoder = joblib.load(MODEL_DIR / 'encoder.joblib')
model = joblib.load(MODEL_DIR / 'model.joblib')

# fill missing values - transform only, NO fit!
for col in CONTINUOUS_FEATURES:
    inference_df[col] = inference_df[col].fillna(inference_df[col].mean())
for col in CATEGORICAL_FEATURES:
    inference_df[col] = inference_df[col].fillna(inference_df[col].mode()[0])

# transform
scaled_inf = pd.DataFrame(scaler.transform(inference_df[CONTINUOUS_FEATURES]), columns=CONTINUOUS_FEATURES, index=inference_df.index)
encoded_inf = pd.DataFrame(encoder.transform(inference_df[CATEGORICAL_FEATURES]), columns=encoder.get_feature_names_out(CATEGORICAL_FEATURES), index=inference_df.index)
X_inference = pd.concat([scaled_inf, encoded_inf], axis=1)

# predict
predictions = np.maximum(model.predict(X_inference), 1)
print(f'Predicted {len(predictions)} house prices')
predictions

Predicted 1459 house prices


array([132627.66301643, 163533.21168884, 164821.95091833, ...,
       155218.8554059 , 129303.65576538, 215506.73510252], shape=(1459,))